# 05 Gold: `daily_brief` (the GenAI narrator)

When the PM opens the app each morning, the chat starts with this **Daily Brief**. It is written by `ai_query('databricks-claude-sonnet-4-5', ...)` from a small context payload built from the gold tables, and then **checked by a validator before anyone sees it**. The point of this notebook is not that an LLM can write a paragraph; it is that every number and ID in the paragraph can be traced to real data, and that the check is code, not a promise.

## What this notebook does

1. **Build the context** from gold tables only (`kpi_daily`, `compliance_findings`, `schedule_overdue`, `risk_forecast`) and store it verbatim in the row's `context_json`. The model sees nothing else.
2. **Prompt** (`brief_lib.build_brief_prompt`, `prompt_version = v2`; v1 was the first run, whose brief said "three milestones are already late" because the context only listed the worst three of seven overdue milestones, so v2 adds the overdue totals to the context and tells the model to count from them): use only CONTEXT facts, reproduce numbers exactly, never invent milestones, drawings, people, dates or causes, describe risk by band only, frame compliance as triage ("N of M drawings flagged for engineer review this week"), call late milestones "already late". Shape: one status sentence, four to five bold-label bullets (Schedule, Budget, Safety, Compliance, Risk), one closing question, about 130 words.
3. **Plan check, then call.** Before each model call the notebook asserts that the query plan has exactly one `AI Query` step, so a call cannot silently fan out. Expect 1 call, at most 2.
4. **Validate** with `brief_lib.validate_brief` (pure Python, unit-tested in `05_gold/tests/test_brief_lib.py`, written test-first). Four checks: every number in the brief matches a number in the context (after normalising `$4.8M`, `$4.8 million`, `2.3%`, commas); every `MS-` and drawing ID exists in the context; no probability language and no percentage next to risk wording; at most 150 words.
5. **One retry, no more.** If validation fails, the failed checks are appended to the prompt and the model is called once more. Both drafts, with their failures, are kept in `validation_detail`. If the second draft also fails the row is still written with `validation_status = 'fail'` and printed prominently below. Nothing is dropped silently, and the prompt is not tuned in a loop.
6. **Write** `mfg_mc_se_sa.bluerridge_gold_pm.daily_brief` (primary key `project_id, brief_date`) with `CREATE OR REPLACE TABLE`, read the row back, re-validate it from the stored `context_json`, and write `outputs/daily_brief_today.md` from that row.

## Decisions worth knowing

- **One brief, for 2026-11-06 only.** `risk_forecast` and `schedule_overdue` are as-of tables: they hold today's view, not a history. A brief for an earlier date would have to mix that date's KPIs with today's risks and overdue list, so it could not be point-in-time. Rather than back-fill briefs that look right but are not, the table holds the one date it can honestly produce. A daily job would add one row per day from here.
- **Risk is shown as a band, never a score.** The risk score is a heuristic: in its backtest the 80 percent interval covered about 14 percent of held-out weeks (target 80), so it is overconfident. `slip_probability` is **not in the context**, so the model cannot quote it; the validator also rejects any percentage near risk wording.
- **The model is told not to calculate.** If the model writes "21% failed" (51 of 238 computed), that number is not in the context and the validator fails it. I chose to keep the validator strict and tell the prompt not to derive new numbers, rather than add derived values to the context.
- **Validator exemptions, each deliberate.** Digits inside milestone and drawing IDs are checked as IDs, not as numbers. Line-start list markers (`1.`) and ordinals (`1st`) are ignored. Money may be rounded to the precision written (`$5M` matches 4.8M; `$4.9M` does not, `$4.8M` does). Plain integers and percentages must match exactly. Spelled-out numbers ("six") are not read, which is why the prompt asks for digits.
- **Context fields.** KPI block: all `kpi_daily` fields of the as-of row except `computed_at`, `is_current` (bookkeeping), with the four `compliance_*_week` counts placed under `compliance_this_week`. "Failed" in `kpi_daily` counts drawings with at least one FAIL finding (51); the finding-level counts (85 findings this week) are labelled as findings. The top 5 failing findings are ordered by severity (critical, high, medium, low), then `drawing_id`, then `finding_id`; each `reason` is cut at a word boundary to 140 characters.

In [1]:
import json, os, sys, contextlib, io, re
from datetime import datetime, timezone
from decimal import Decimal
import datetime as dt
from pathlib import Path
from pyspark.sql import functions as F

sys.path.insert(0, os.getcwd())   # a notebook's working directory is its own folder, where brief_lib.py lives
import brief_lib as bl

dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
dbutils.widgets.text("as_of_date", "2026-11-06", "As-of date")
dbutils.widgets.text("run_id", "", "Job run id (set by the job as {{job.run_id}})")
catalog = dbutils.widgets.get("catalog"); prefix = dbutils.widgets.get("schema_prefix")
AS_OF = dbutils.widgets.get("as_of_date"); RUN_ID_PARAM = dbutils.widgets.get("run_id")
GOLD = f"{catalog}.{prefix}_gold_pm"
PROJECT = "BRL-PH2"
MODEL = "databricks-claude-sonnet-4-5"
WEEK_START = (dt.date.fromisoformat(AS_OF) - dt.timedelta(days=6)).isoformat()   # 2026-10-31 for 2026-11-06
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

job_run_id = RUN_ID_PARAM
if not job_run_id or "{{" in job_run_id:
    try: job_run_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().toString()
    except Exception: job_run_id = "unavailable"
print("job_run_id:", job_run_id, "| raw parameter:", repr(RUN_ID_PARAM))
print("as_of", AS_OF, "| compliance week", WEEK_START, "to", AS_OF, "| model", MODEL, "| prompt_version", bl.PROMPT_VERSION)

job_run_id: 438698467007318 | raw parameter: '438698467007318'
as_of 2026-11-06 | compliance week 2026-10-31 to 2026-11-06 | model databricks-claude-sonnet-4-5 | prompt_version v2


## 1. Build the context (gold tables only)

Every value below comes from a SELECT on a gold table. `slip_probability` is deliberately never selected.

In [2]:
def py(v):
    if isinstance(v, Decimal): return float(v)
    if isinstance(v, (dt.date, dt.datetime)): return v.isoformat()
    if hasattr(v, "item"): return v.item()      # numpy scalar
    return v

def rows(sql):
    return [{k: py(v) for k, v in r.asDict().items()} for r in spark.sql(sql).collect()]

k = rows(f"SELECT * FROM {GOLD}.kpi_daily WHERE project_id = '{PROJECT}' AND snapshot_date = DATE'{AS_OF}'")
assert len(k) == 1 and k[0]["is_current"] is True, "expected exactly one current kpi_daily row for the as-of date"
k = k[0]
COMPLIANCE_KEYS = {"compliance_reviewed_week": "reviewed_drawings", "compliance_passed_week": "passed_drawings",
                   "compliance_failed_week": "flagged_drawings", "compliance_review_week": "review_unusable_drawings"}
kpi = {c: v for c, v in k.items() if c not in COMPLIANCE_KEYS and c not in ("computed_at", "is_current")}

WEEK = f"review_date BETWEEN DATE'{WEEK_START}' AND DATE'{AS_OF}'"
def short(s, n=140):
    s = " ".join(str(s).split())
    return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + "..."
top_fail = rows(f'''SELECT drawing_id, citation, severity, reason FROM {GOLD}.compliance_findings
  WHERE status = 'FAIL' AND {WEEK}
  ORDER BY CASE severity WHEN 'critical' THEN 0 WHEN 'high' THEN 1 WHEN 'medium' THEN 2 ELSE 3 END, drawing_id, finding_id LIMIT 5''')
for r in top_fail: r["reason"] = short(r["reason"])
by_cit = rows(f'''SELECT citation, count(*) AS fail_findings FROM {GOLD}.compliance_findings WHERE status = 'FAIL' AND {WEEK}
  GROUP BY citation ORDER BY fail_findings DESC, citation''')
fail_total = rows(f"SELECT count(*) AS n FROM {GOLD}.compliance_findings WHERE status = 'FAIL' AND {WEEK}")[0]["n"]
overdue = rows(f'''SELECT milestone_id, name, days_late, downstream_open_count FROM {GOLD}.schedule_overdue
  ORDER BY criticality DESC, days_late DESC, milestone_id LIMIT 3''')
ov_total = rows(f"SELECT count(*) AS n, count_if(days_late > 0) AS late, count_if(days_late = 0) AS today FROM {GOLD}.schedule_overdue")[0]
risk = rows(f'''SELECT milestone_id, name, discipline, planned_end, risk_band FROM {GOLD}.risk_forecast ORDER BY risk_rank LIMIT 3''')

context = {
    "project": "Blue Ridge LNG Terminal, Phase 2",
    "as_of_date": AS_OF,
    "kpi": kpi,
    "compliance_this_week": {
        "window": f"{WEEK_START} to {AS_OF}",
        **{new: k[old] for old, new in COMPLIANCE_KEYS.items()},
        "fail_findings_total": fail_total,
        "top_fail_findings": top_fail,
        "fail_findings_by_citation": by_cit,
    },
    "schedule_overdue_total": ov_total["n"],
    "schedule_already_late": ov_total["late"],
    "schedule_due_today": ov_total["today"],
    "schedule_overdue_top3": overdue,
    "risk_forecast_top3": risk,
}
context_json = json.dumps(context, indent=2, sort_keys=True)
assert "slip_probability" not in context_json
print(context_json)

{
  "as_of_date": "2026-11-06",
  "compliance_this_week": {
    "fail_findings_by_citation": [
      {
        "citation": "ASME B31.3 \u00a7322",
        "fail_findings": 30
      },
      {
        "citation": "OSHA 1910.147",
        "fail_findings": 25
      },
      {
        "citation": "ASME B16.5 \u00a74.2",
        "fail_findings": 15
      },
      {
        "citation": "IBC 1005.1",
        "fail_findings": 8
      },
      {
        "citation": "ASME B31.3 \u00a7341",
        "fail_findings": 7
      }
    ],
    "fail_findings_total": 85,
    "flagged_drawings": 51,
    "passed_drawings": 187,
    "review_unusable_drawings": 0,
    "reviewed_drawings": 238,
    "top_fail_findings": [
      {
        "citation": "ASME B31.3 \u00a7322",
        "drawing_id": "BRL-INS-PID-0018",
        "reason": "Vessel V-2102 (SUCTION DRUM) has no pressure relief device shown. General Note 4 requires every pressure vessel to be protected by a PSV...",
        "severity": "critical"
      },

## 2. Prompt, plan check and the model call

`call_model` puts the prompt in a one-row DataFrame and applies `ai_query` to it. Before running, it reads the physical plan and asserts exactly one `AI Query` step (`explain` only plans, it runs nothing). Only the physical plan is read, because the Photon section that follows quotes the step again.

In [3]:
PLANS = []
def call_model(prompt, label):
    df = spark.createDataFrame([(prompt,)], "prompt string").select(F.expr(f"ai_query('{MODEL}', prompt)").alias("draft"))
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode="formatted")
    plan = re.split(r"^== (?!Physical Plan)", buf.getvalue(), flags=re.M)[0]
    steps = re.findall(r"^\(\d+\) AI Query.*$", plan, flags=re.M)
    print(f"[{label}] AI Query steps in the physical plan: {len(steps)}")
    assert len(steps) == 1, "the call must plan exactly one AI Query step"
    PLANS.append((label, len(steps)))
    return df.collect()[0]["draft"].strip()

prompt_v1 = bl.build_brief_prompt(context)
print("PROMPT (attempt 1), verbatim:\n" + "=" * 70)
print(prompt_v1)
print("=" * 70, "| prompt words:", len(prompt_v1.split()))

PROMPT (attempt 1), verbatim:
You are the project manager's co-pilot on the Blue Ridge LNG Terminal, Phase 2 build. Write today's Daily Brief for the project manager from the CONTEXT below.

GROUNDING RULES
- Use only facts in the CONTEXT. Never invent milestones, drawings, people, dates or causes.
- Reproduce numbers exactly as given, as digits. Rounding to the precision shown is fine (for example $4.8M). Do not calculate new numbers (no percentages of your own, no sums, no differences).
- Describe schedule risk only by its band ("High risk", "Medium risk"). Never write a percentage, probability, chance or likelihood for risk.
- Frame compliance as triage: "N of M drawings flagged for engineer review this week". Do not say the drawings are non-compliant or failed.
- Milestones listed under schedule_overdue_top3 are already late: say "already late". schedule_overdue_top3 shows only the worst three; when you say how many milestones are late or overdue, use the totals (schedule_overdue_t

## 3. Attempt 1, validate, and at most one retry

In [4]:
attempts = []
def run_attempt(n, prompt):
    draft = call_model(prompt, f"attempt {n}")
    ok, failures = bl.validate_brief(draft, context)
    attempts.append({"attempt": n, "draft": draft, "ok": ok, "failures": failures, "words": len(draft.split())})
    print(f"--- attempt {n}: validator ok={ok}, words={len(draft.split())}")
    print(draft)
    for f in failures: print("   FAILED CHECK:", f)
    return draft, ok, failures

draft, ok, failures = run_attempt(1, prompt_v1)
if not ok:
    print("\nValidation failed; one retry with the failed checks appended to the prompt.")
    retry_prompt = bl.build_retry_prompt(context, draft, failures)
    print("RETRY PROMPT SUFFIX, verbatim:\n" + retry_prompt[len(prompt_v1):])
    draft, ok, failures = run_attempt(2, retry_prompt)

status = ("pass" if len(attempts) == 1 else "pass_after_retry") if ok else "fail"
print("\nFINAL validation_status:", status, "| attempts:", len(attempts))
if status == "fail":
    print("!" * 70)
    print("!!! DAILY BRIEF FAILED GROUNDING VALIDATION AFTER ONE RETRY. The row is written and flagged 'fail'.")
    for f in failures: print("!!!  ", f)
    print("!" * 70)
assert sum(n for _, n in PLANS) == len(attempts) <= 2, "model calls must equal attempts and never exceed 2"

[attempt 1] AI Query steps in the physical plan: 1
--- attempt 1: validator ok=True, words=144
Blue Ridge LNG Terminal Phase 2 remains on track overall, though 7 milestones are now overdue and budget variance has reached $4.8M.

**Schedule**: 7 milestones are already late; MS-271 (heat exchanger install) is 15 days overdue with 60 downstream tasks at risk. MS-281 (Tank T-203) slipped 4 days this period. 3 instrumentation milestones (MS-335, MS-351, MS-352) show High risk through January.

**Budget**: Current overrun stands at $4.8M against the $208.7M total budget (2.3% EAC).

**Safety**: 28 days without a recordable incident; 0.0 TRIR over the past 28 days.

**Compliance**: 51 of 238 drawings flagged for engineer review this week. 85 findings led by ASME B31.3 §322 (30 findings, primarily missing pressure relief devices on vessels) and OSHA 1910.147 (25 findings).

**Risk**: 24 open NCRs (9 compliance, 15 workmanship).

Would you like me to drill into the MS-271 delay or the pressure 

## 4. Write `daily_brief` (primary key `project_id, brief_date`)

`CREATE OR REPLACE TABLE` (never drop and create). `validation_detail` holds the checks run and both drafts with their failures, so a failed first attempt stays on record even when the retry passes.

In [5]:
detail = {"job_run_id": job_run_id, "checks": ["numbers match context", "milestone and drawing IDs exist in context", "no probability language or risk percentages", "at most 150 words"],
          "attempts": attempts}
spark.sql(f'''
CREATE OR REPLACE TABLE {GOLD}.daily_brief (
  project_id STRING NOT NULL COMMENT 'Project key, BRL-PH2',
  brief_date DATE NOT NULL COMMENT 'The as-of date the brief describes. One brief per date; only the as-of date is generated because risk_forecast and schedule_overdue are as-of only',
  brief_md STRING COMMENT 'The Daily Brief in Markdown: the final draft, even when validation_status is fail',
  context_json STRING COMMENT 'The exact context payload the model was given, built from gold tables only. Every number and ID in brief_md must trace to it',
  model STRING COMMENT 'Serving endpoint used by ai_query',
  prompt_version STRING COMMENT 'Version of the prompt in 05_gold/brief_lib.py',
  validation_status STRING COMMENT 'pass | pass_after_retry | fail, from brief_lib.validate_brief',
  validation_detail STRING COMMENT 'JSON: checks run, job run id, and every attempt with its draft, ok flag and failed checks',
  attempts INT COMMENT 'Model calls made: 1, or 2 after the single retry',
  generated_at TIMESTAMP COMMENT 'When the row was written',
  CONSTRAINT daily_brief_pk PRIMARY KEY (project_id, brief_date))
COMMENT 'The Daily Brief the app opens the chat with: an ai_query narrative over a gold-table context payload, validated for grounding (numbers, IDs, no probabilities, length) with one retry. Risk is described by band only.'
''')
row = spark.createDataFrame([(PROJECT, dt.date.fromisoformat(AS_OF), draft, context_json, MODEL, bl.PROMPT_VERSION, status,
                              json.dumps(detail, indent=2), len(attempts))],
                            "project_id string, brief_date date, brief_md string, context_json string, model string, prompt_version string, validation_status string, validation_detail string, attempts int")
row.withColumn("generated_at", F.current_timestamp()).createOrReplaceTempView("_brief_row")
spark.sql(f"INSERT INTO {GOLD}.daily_brief SELECT project_id, brief_date, brief_md, context_json, model, prompt_version, validation_status, validation_detail, attempts, generated_at FROM _brief_row")
print("wrote", GOLD + ".daily_brief")

wrote mfg_mc_se_sa.bluerridge_gold_pm.daily_brief


## 5. Read the row back and re-validate it from what was stored

In [6]:
t = spark.table(f"{GOLD}.daily_brief")
assert t.count() == 1
r = t.collect()[0].asDict()
display(t.select("project_id", "brief_date", "model", "prompt_version", "validation_status", "attempts", "generated_at"))
print("columns:", t.columns)
assert (r["project_id"], str(r["brief_date"])) == (PROJECT, AS_OF) and r["validation_status"] in ("pass", "pass_after_retry", "fail")
stored_context = json.loads(r["context_json"])
assert stored_context == context, "context_json must be the payload the model saw"
ok2, failures2 = bl.validate_brief(r["brief_md"], stored_context)
assert ok2 == (r["validation_status"] != "fail"), "re-validating the stored row must agree with the stored status"
words = len(r["brief_md"].split())
print(f"re-validation of the stored row: ok={ok2}, failures={failures2}, words={words}")
print("\nDAILY BRIEF for", AS_OF, "(validation_status =", r["validation_status"] + ")\n" + "-" * 70)
print(r["brief_md"])
print("-" * 70)
if r["validation_status"] == "fail":
    print("!!! THIS BRIEF FAILED VALIDATION:", failures2)

project_id,brief_date,model,prompt_version,validation_status,attempts,generated_at
BRL-PH2,2026-11-06,databricks-claude-sonnet-4-5,v2,pass,1,2026-11-08T17:19:00.914Z


columns: ['project_id', 'brief_date', 'brief_md', 'context_json', 'model', 'prompt_version', 'validation_status', 'validation_detail', 'attempts', 'generated_at']
re-validation of the stored row: ok=True, failures=[], words=144

DAILY BRIEF for 2026-11-06 (validation_status = pass)
----------------------------------------------------------------------
Blue Ridge LNG Terminal Phase 2 remains on track overall, though 7 milestones are now overdue and budget variance has reached $4.8M.

**Schedule**: 7 milestones are already late; MS-271 (heat exchanger install) is 15 days overdue with 60 downstream tasks at risk. MS-281 (Tank T-203) slipped 4 days this period. 3 instrumentation milestones (MS-335, MS-351, MS-352) show High risk through January.

**Budget**: Current overrun stands at $4.8M against the $208.7M total budget (2.3% EAC).

**Safety**: 28 days without a recordable incident; 0.0 TRIR over the past 28 days.

**Compliance**: 51 of 238 drawings flagged for engineer review this week.

## 6. Evidence file `outputs/daily_brief_today.md`

Written from the table row (not from the in-memory draft). The brief comes first, then how it was grounded, then the context JSON in a collapsed block. Each check line is derived from the validator's failure list for the stored brief.

In [7]:
def check_line(name, matcher):
    bad = [f for f in failures2 if matcher(f)]
    return f"- {name}: " + ("PASS" if not bad else "FAIL (" + "; ".join(bad) + ")")
lines = [
    check_line("Numbers: every number matches a number in the context", lambda f: f.startswith("number ")),
    check_line("IDs: every milestone and drawing ID exists in the context", lambda f: " ID " in f),
    check_line("No probabilities: no probability wording, no percentage next to risk wording", lambda f: "probab" in f.lower()),
    check_line(f"Length: {words} words (limit {bl.MAX_WORDS})", lambda f: f.startswith("too long")),
]
md_text = f'''# Daily Brief, {r["brief_date"]}

{r["brief_md"]}

## How this was grounded

- Validation status: **{r["validation_status"]}** ({r["attempts"]} model call{"s" if r["attempts"] != 1 else ""}; a failed first draft would be kept in `validation_detail`)
- Model: `{r["model"]}`, prompt version `{r["prompt_version"]}`
- Generated at: {r["generated_at"]} UTC, job run `{job_run_id}`
- Source: `{GOLD}.daily_brief`, key `({r["project_id"]}, {r["brief_date"]})`
- Validator checks (`05_gold/brief_lib.py`, `validate_brief`), run on the stored brief against the stored context:
''' + "\n".join("  " + l for l in lines) + f'''
- Risk is shown by band only. The heuristic risk score is not in the context, so the model could not quote it.
- Compliance is triage: drawings flagged for engineer review, not a verdict.

<details><summary>Context JSON the model was given</summary>

```json
{r["context_json"]}
```

</details>
'''
(OUT / "daily_brief_today.md").write_text(md_text)
print((OUT / "daily_brief_today.md").stat().st_size, "bytes written to outputs/daily_brief_today.md")
print(md_text[:1800])
print("plan checks:", PLANS)

5936 bytes written to outputs/daily_brief_today.md
# Daily Brief, 2026-11-06

Blue Ridge LNG Terminal Phase 2 remains on track overall, though 7 milestones are now overdue and budget variance has reached $4.8M.

**Schedule**: 7 milestones are already late; MS-271 (heat exchanger install) is 15 days overdue with 60 downstream tasks at risk. MS-281 (Tank T-203) slipped 4 days this period. 3 instrumentation milestones (MS-335, MS-351, MS-352) show High risk through January.

**Budget**: Current overrun stands at $4.8M against the $208.7M total budget (2.3% EAC).

**Safety**: 28 days without a recordable incident; 0.0 TRIR over the past 28 days.

**Compliance**: 51 of 238 drawings flagged for engineer review this week. 85 findings led by ASME B31.3 §322 (30 findings, primarily missing pressure relief devices on vessels) and OSHA 1910.147 (25 findings).

**Risk**: 24 open NCRs (9 compliance, 15 workmanship).

Would you like me to drill into the MS-271 delay or the pressure relief device fin